# Testing scipy linear programming

## Imports, definitions and base tests

In [1]:
# Module imports
import sys

import numpy as np
import scipy as sp
from loguru import logger



In [2]:
# Set up the logger
logger.remove()
logger.add(
    sink=sys.stdout,
    format="<level>{level:<10} | {message}</>",
    level="INFO",
    colorize=True,
)

1

### Behaviors as vectors and notation

Suppose a (2,2,2) routed Bell experiment.

$ p(ab|xyz) \in \mathbb{R}^{32}$ is the observed behavior, assumed to be no-signaling : $$p \in \mathcal{NS}$$

We wish for easy conversion from a vector format (for algebraic operations) to a matrix format (for leggibility) :
$$p=\begin{pmatrix} p_{00|00S} \\ p_{00|01S} \\ \vdots \\ p_{00|00L} \\ \vdots \end{pmatrix}\quad \leftrightarrow \quad p=\begin{pmatrix} p_{00|00}& p_{00|01}& p_{00|10}& p_{00|11}\\ p_{01|00}& p_{01|01}& \dots\\ \vdots & & \ddots \\ & & & p_{11|11} \end{pmatrix}$$
where, to account for both matrices $p(z=S)$ and $p(z=L)$, both cases are represented in two matrices, making $p$ either a column vector in $\mathbb{R}^{32}$ or a third-order tensor of shape $(2,4,4)$.

We defined in the Behavior class (behavior.py) some utility functions.

In [3]:
from behaviors import Behavior

# The maximally mixed state over the experiment space
I = Behavior((1 / 4) * np.ones(32))  # noqa: E741

# The usual (2,2,2) PR box
SR_pr_box = np.array(
    [ 1/2, 1/2, 1/2, 0, 0, 0, 0, 1/2, 0, 0, 0, 1/2, 1/2, 1/2, 1/2, 0]
)

# The PR box in the experiment space : p(ab|xy) is assumed to be
# independent of the value of z
pr_box = Behavior(np.concatenate((SR_pr_box, SR_pr_box), axis=0))

def to_behavior(arr):
    return Behavior(np.concatenate((arr, arr), axis=0))

non_positive_arr = np.array(
    [-1/2,-1/2,-1/2,0,0,0,0,1/2,0,0,0,1/2,1/2,1/2,1/2,0,],
)
non_normalized_arr = np.array(
    [1/2,1/2,1/2,0,0,0,0,0,0,0,0,1/2,1/2,1/2,1/2,0,]
)
non_ns_arr = np.array(
    [1/2,1/2,0,0,0,0,1/2,1/2,1/2,1/2,0,0,0,0,1/2,1/2,]
)

non_positive = to_behavior(non_positive_arr)
non_normalized = to_behavior(non_normalized_arr)
non_ns = to_behavior(non_ns_arr)


In [4]:
# Sanity check cell, don't mind me

# print("PR box: ", pr_box)
# print("I: ", I)

print("I == I: ", I == I)
print("I == pr_box: ", I == pr_box)
print("I == 0: ", I == 0)
print("I == 0.25: ", I == 0.25)
print("pr_box == pr_box_array: ", pr_box == np.concatenate((SR_pr_box, SR_pr_box), axis=0))

print("I positive: ", I.positivity())
print("I normalized: ", I.normalization())
print("I no-signaling: ", I.no_signaling())

print("PR box positive: ", pr_box.positivity())
print("PR box normalized: ", pr_box.normalization())
print("PR box no-signaling: ", pr_box.no_signaling())

print("Non-positive: ", non_positive.positivity())
print("Non-normalized: ", non_normalized.normalization())
print("Non-no-signaling: ", non_ns.no_signaling())

print("Non-no-signaling is normalized: ", non_ns.is_normalized())

I == I:  True
I == pr_box:  False
I == 0:  False
I == 0.25:  True
pr_box == pr_box_array:  True
I positive:  True
I normalized:  True
I no-signaling:  True
PR box positive:  True
PR box normalized:  True
PR box no-signaling:  True
Non-positive:  False
Non-normalized:  False
Non-no-signaling:  False
Non-no-signaling is normalized:  True


In [5]:
# Checking the indices correspondence functions

from behaviors import routed_index_to_indices, routed_indices_to_index

for i in range(2*2**2*2**2):
    print(f"i={i} -> routed index: {routed_index_to_indices(i, m=2)}, computed i: {routed_indices_to_index(*routed_index_to_indices(i,m=2), m=2)}")  # noqa: E501

i=0 -> routed index: (0, 0, 0, 0, 0), computed i: 0
i=1 -> routed index: (0, 0, 0, 1, 0), computed i: 1
i=2 -> routed index: (0, 0, 1, 0, 0), computed i: 2
i=3 -> routed index: (0, 0, 1, 1, 0), computed i: 3
i=4 -> routed index: (0, 1, 0, 0, 0), computed i: 4
i=5 -> routed index: (0, 1, 0, 1, 0), computed i: 5
i=6 -> routed index: (0, 1, 1, 0, 0), computed i: 6
i=7 -> routed index: (0, 1, 1, 1, 0), computed i: 7
i=8 -> routed index: (1, 0, 0, 0, 0), computed i: 8
i=9 -> routed index: (1, 0, 0, 1, 0), computed i: 9
i=10 -> routed index: (1, 0, 1, 0, 0), computed i: 10
i=11 -> routed index: (1, 0, 1, 1, 0), computed i: 11
i=12 -> routed index: (1, 1, 0, 0, 0), computed i: 12
i=13 -> routed index: (1, 1, 0, 1, 0), computed i: 13
i=14 -> routed index: (1, 1, 1, 0, 0), computed i: 14
i=15 -> routed index: (1, 1, 1, 1, 0), computed i: 15
i=16 -> routed index: (0, 0, 0, 0, 1), computed i: 16
i=17 -> routed index: (0, 0, 0, 1, 1), computed i: 17
i=18 -> routed index: (0, 0, 1, 0, 1), computed 

In [6]:
# Checking that the no-signaling set is well defined in no_signaling_set.py

from no_signaling_set import routed_no_signaling_equations
from behaviors import completely_mixed_behavior, pr_box

delta, m = 2, 2
equations, right_side, dimensionality = routed_no_signaling_equations(delta, m)


# print(f"Delta: {delta}, m: {m}")
# print("\n--------------\n")
# print(f"Equations shape: {equations.shape}")
# print("\n--------------\n")

# for line in equations:
#     print(str(line).strip("[]").replace("\n", "").replace(" ", "").replace("-1", "2").replace(".", "").replace("0", "."))  # noqa: E501

# print("\n--------------\n")
# print(f"Right side shape: {right_side.shape}")
# print("\n--------------\n")
# print(f"Right side: {right_side}")


print(np.all(equations @ completely_mixed_behavior.get_vector() == right_side))
print(np.all(equations @ pr_box.get_vector() == right_side))



True
True


## Sampling behaviors

In [7]:
from samplers import UniformNormalizedSampler

sampler = UniformNormalizedSampler(2,2,True) # Samples normalized behaviors

sample_behavior = sampler.sample()
print("Sampled behavior: ", sample_behavior)
print("Sample behavior is normalized: ", sample_behavior.is_normalized())
print("Sample behavior is no-signaling: ", sample_behavior.no_signaling())

Sampled behavior:  Behavior:
Short path (z=S):
[[0.30037236 0.16010417 0.08931082 0.04198749]
 [0.43925898 0.23714764 0.00314626 0.34671485]
 [0.03089501 0.48234995 0.4703636  0.42611375]
 [0.22947366 0.12039823 0.43717933 0.18518391]]
Long path (z=L) :
[[0.33441458 0.16410436 0.69733077 0.17505671]
 [0.24760153 0.77156928 0.19507434 0.64470888]
 [0.0347166  0.03119943 0.07518457 0.15097604]
 [0.38326728 0.03312693 0.03241032 0.02925837]]
------------
Sample behavior is normalized:  True
Sample behavior is no-signaling:  False


Sampling no-signaling behaviors can't reasonably be achieved with rejection sampling from normalized behaviors though, since as $dim(\mathcal{NS}) < dim(\mathcal{B})$, the usual measure of $\mathcal{NS}$ in $\mathcal{B}$ is null. Getting a no-signaling behavior would thus be very, very lucky.

A consequence of this is that we will need to implement uniform sampling on the $\mathcal{NS}$ polytope to sample no-signaling behaviors directly. This can be achieved if we know the polytope's vertices, which is the case in low-dimensions only. The method consists in partitioning the arbitrary bounded polytope in simplices, which we can sample from by weighing them using their volumes, and then using simplex-specific methods to sample uniformly in the chosen simplex.

It also seems that we are able to uniformly sample from a polytope using MCMC methods, as per [Sun and Chen, 2024](https://arxiv.org/abs/2412.06629), using the ``polytopewalk`` module.

### Testing ``polytopewalk``

In [14]:
import polytopewalk as pw

help(pw.sparse)

Help on module polytopewalk.sparse in polytopewalk:

NAME
    polytopewalk.sparse - Sparse Module

CLASSES
    pybind11_builtins.pybind11_object(builtins.object)
        SparseCenter
        SparseRandomWalk
            SparseBallWalk
            SparseBarrierWalk
                SparseDikinLSWalk
                SparseDikinWalk
                SparseJohnWalk
                SparseVaidyaWalk
            SparseHitAndRun

    class SparseBallWalk(SparseRandomWalk)
     |  Sparse Ball Walk Implementation.
     |
     |  Method resolution order:
     |      SparseBallWalk
     |      SparseRandomWalk
     |      pybind11_builtins.pybind11_object
     |      builtins.object
     |
     |  Methods defined here:
     |
     |  __init__(...)
     |      __init__(self: polytopewalk.sparse.SparseBallWalk, r: float = 0.5, thin: int = 1) -> None
     |
     |
     |      Initialization for Sparse Ball Walk Class.
     |      Runs on Constrained Polytope Form: Ax = b, x >=_k 0.
     |
     |      P

In [21]:
import inspect
import importlib
import pydoc
import types
import re

def extract_full_docs(module_name, output_file="module_docs.txt", max_depth=2):
    try:
        module = importlib.import_module(module_name)
    except ImportError:
        print(f"Could not import module '{module_name}'")
        return

    visited = set()

    def write_header(f, name, level):
        header = f"\n{'=' * (80 - 2*level)}\n{name}\n{'=' * (80 - 2*level)}\n"
        f.write(header)

    def walk(obj, name, f, level=0):
        if id(obj) in visited or level > max_depth:
            return
        visited.add(id(obj))

        write_header(f, name, level)
        doc = pydoc.render_doc(obj) or "(No docstring found)"
        f.write(re.sub(r'.\x08','',doc) + "\n\n")

        if isinstance(obj, types.ModuleType) or inspect.isclass(obj):
            try:
                for member_name, member in inspect.getmembers(obj):
                    # Skip builtins, private, or huge base objects
                    if member_name.startswith("__") and member_name.endswith("__"):
                        continue
                    # Don't go too deep into imported modules
                    if inspect.ismodule(member) and member.__name__.split('.')[0] != module_name.split('.')[0]:
                        continue
                    full_name = f"{name}.{member_name}"
                    walk(member, full_name, f, level + 1)
            except Exception as e:
                f.write(f"\n[Error while inspecting {name}: {e}]\n")

    with open(output_file, "w", encoding="utf-8") as f:
        walk(module, module_name, f)

    print(f"Documentation safely written to '{output_file}' ✅")

# Example usage:


name = "polytopewalk"
extension = ".txt"

extract_full_docs(name, output_file=f"{name}_doc{extension}", max_depth=5)


Documentation safely written to 'polytopewalk_doc.txt' ✅


In [ ]:
A, b, k = routed_no_signaling_equations(delta=2, m=2)

other_k = A.shape[1]

A = sp.sparse.csc_matrix(A, dtype=np.float64)
b = b.astype(np.float64).reshape(-1, 1)

walk = pw.sparse.SparseHitAndRun()
fr = pw.FacialReduction()
sc = pw.sparse.SparseCenter()

points = pw.sparseFullWalkRun(A, b, other_k, 3, walk, fr, sc)

print("Point: ", points[0])

# Check that the point is in the no-signaling set
# sampled_behavior = Behavior(points[0])

Point:  [ 0.36099237  0.35434677  0.55866032  0.53321905 -0.08202642 -0.07538082
 -0.09450662 -0.06906534  0.60198013  0.50766638  0.40431218  0.32879411
  0.11905391  0.21336767  0.13153411  0.20705219  0.2160583   0.07011761
  0.30884294  0.07661048  0.06290765  0.20884834  0.15531077  0.38754323
  0.39663376  0.44590836  0.30384913  0.4394155   0.32440028  0.27512568
  0.23199716  0.09643079]
